# Phase 1: Colab Environment Setup

Set up the full SoARM + LIBERO + OpenVLA-OFT stack on a fresh Colab A100 runtime.

## Requirements

- [ ] ENV-01: All dependencies install in correct order without pip resolver conflicts
- [ ] ENV-02: EGL headless rendering configured — OffScreenRenderEnv produces non-black LIBERO frames
- [ ] ENV-03: OpenVLA-OFT loads on GPU (A100 bf16) and returns a valid 7-D action tensor

## Usage

**BLOCK A** (this block): Run cells 0-9 top to bottom, then restart the runtime.

**BLOCK B** (Plan 02): After restart, run verification cells for ENV-01 / ENV-02 / ENV-03.

> Note: Block A must complete fully before restarting. Do not run Block B cells before restart.

In [1]:
import os

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set REPO_ROOT to the path where you cloned SoARM-Research.
# If using Google Drive: "/content/drive/MyDrive/SoARM-Research"
# If using git clone directly to Colab: "/content/SoARM-Research"
REPO_ROOT = "/content/drive/MyDrive/SoARM-Research"
# ────────────────────────────────────────────────────────────────────────────

# Derived path constants (do not edit these)
LIBERO_ROOT = f"{REPO_ROOT}/LIBERO/libero/libero"
LIBERO_PKG  = f"{REPO_ROOT}/LIBERO"   # path to setup.py directory
OUT_DIR     = f"{REPO_ROOT}/LIBERO/notebooks/outputs"
BDDL_FILE   = (
    f"{LIBERO_ROOT}/bddl_files/libero_spatial/"
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl"
)

# Create outputs directory so Block B render check can save there
os.makedirs(OUT_DIR, exist_ok=True)

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"LIBERO_ROOT = {LIBERO_ROOT}")
print(f"LIBERO_PKG  = {LIBERO_PKG}")
print(f"OUT_DIR     = {OUT_DIR}")
print(f"BDDL_FILE   = {BDDL_FILE}")
print(f"Saved → {OUT_DIR}  (outputs directory ready)")

REPO_ROOT   = /content/drive/MyDrive/SoARM-Research
LIBERO_ROOT = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero
LIBERO_PKG  = /content/drive/MyDrive/SoARM-Research/LIBERO/libero
OUT_DIR     = /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs
BDDL_FILE   = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero/bddl_files/libero_spatial/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl
Saved → /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs  (outputs directory ready)


In [2]:
# GPU assertion — D-06
# Check GPU availability and warn loudly if not A100.
# OpenVLA-OFT in bf16 requires ~16 GB VRAM; A100 (40 GB) is the target.
import torch

assert torch.cuda.is_available(), (
    "No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if "A100" not in gpu_name:
    print()
    print("WARNING: Expected A100, got", gpu_name)
    print("WARNING: OpenVLA-OFT in bf16 requires ~16 GB+ VRAM.")
    print("WARNING: ENV-03 will OOM on T4 (15 GB). Restart with an A100 runtime.")
    print("WARNING: You may still proceed for install-only testing on T4.")
else:
    print("A100 confirmed. Proceeding.")

AssertionError: No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU.

---

## BLOCK A: Install

Run all cells in this block **top to bottom**, then restart the runtime.

**Ordering is critical** — do not reorder or skip cells:

1. EGL system packages (apt) must install before pip mujoco
2. PyTorch must install before flash-attn (flash-attn compiles against torch CUDA headers)
3. The custom transformers fork must install last (prevents pip downgrade to PyPI version)

---

In [3]:
# Step 1 of 6 — EGL system packages
# Must run BEFORE pip mujoco install.
# These C libraries must exist when the mujoco Python extension builds.
# apt-get update refreshes repo index — prevents 404 for stale package URLs (e.g. libosmesa6).
!apt-get update -qq
!apt-get install -y -q --fix-missing \
    libglfw3 \
    libglew-dev \
    libosmesa6-dev \
    libgles2 \
    libglvnd0 \
    libegl-dev \
    libegl1 \
    libgl1-mesa-glx

Reading package lists...
Building dependency tree...
Reading state information...
libegl1 is already the newest version (1.4.0-1).
libgles2 is already the newest version (1.4.0-1).
libglvnd0 is already the newest version (1.4.0-1).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
The following additional packages will be installed:
  libgl-dev libglew2.2 libglu1-mesa libglu1-mesa-dev libglx-dev libosmesa6
Suggested packages:
  glew-utils libgles1 libvulkan1
The following NEW packages will be installed:
  libegl-dev libgl-dev libglew-dev libglew2.2 libglfw3 libglu1-mesa
  libglu1-mesa-dev libglx-dev libosmesa6 libosmesa6-dev
0 upgraded, 10 newly installed, 0 to remove and 3 not upgraded.
Need to get 4,199 kB of archives.
After this operation, 19.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglx-dev amd64 1.4.0-1 [14.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgl-dev amd64 1.4.0-1 [101 k

In [4]:
# Step 2 of 6 — PyTorch 2.2.0 (cu121)
# Must run BEFORE flash-attn: flash-attn compiles CUDA kernels against installed torch headers.
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 700.7 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 72.3 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.1 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 39.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 33.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 59.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB ? eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━

In [5]:
# Step 3 of 6 — MuJoCo + simulation stack
# Exact versions from LIBERO/requirements.txt.
# robosuite must be 1.4.0 — 1.5.x removed SingleArmEnv which LIBERO depends on.
!pip install mujoco==2.3.7 gym==0.25.2 -q
!pip install \
    robosuite==1.4.0 \
    bddl==1.0.1 \
    easydict==1.9 \
    cloudpickle==2.1.0 \
    einops==0.4.1 \
    numpy==1.22.4 \
    opencv-python==4.6.0.66 \
    "imageio[ffmpeg]" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 593.3/593.3 kB 8.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 19.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for mujoco
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (mujoco)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 4.3 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 4.7 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit 

In [6]:
# Step 4 of 6 — LIBERO editable install
# editable install — LIBERO package changes are live without reinstall
# LIBERO_PKG is defined in cell 1 (REPO_ROOT config cell).
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", LIBERO_PKG, "-q"],
    capture_output=True,
    text=True,
)
print(result.stdout[-500:] if result.stdout else "(no stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

(no stdout)
STDERR: ERROR: /content/drive/MyDrive/SoARM-Research/LIBERO/libero is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).



In [7]:
# Step 5 of 6 — OpenVLA-OFT supporting packages + custom transformers fork
# The git fork (moojink/transformers-openvla-oft) MUST be installed LAST in this cell.
# Do NOT install transformers from PyPI — the fork replaces it entirely.
# The fork adds bidirectional attention for parallel decoding; PyPI version lacks this.
!pip install \
    timm==0.9.10 \
    tokenizers==0.19.1 \
    sentencepiece==0.1.99 \
    peft==0.11.1 \
    accelerate \
    "huggingface_hub>=1.2.0" -q

# Install transformers fork LAST — pip resolver cannot downgrade to PyPI version this way.
# Commit SHA comment for reproducibility: installs from main branch of the fork repo.
# To pin a specific commit: git+https://github.com/moojink/transformers-openvla-oft.git@<SHA>
!pip install git+https://github.com/moojink/transformers-openvla-oft.git -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 38.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.7 MB/s eta

In [8]:
# Step 6 of 6 — flash-attn (must be last install)
# This cell takes ~10 min to compile CUDA kernels. Do not interrupt.
# Must run AFTER torch is installed: flash-attn compiles against torch CUDA headers.
!pip install packaging ninja -q
!pip install "flash-attn==2.5.5" --no-build-isolation -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.8 MB/s eta 0:00:0000:0100:01
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


---

## *** STOP — Restart runtime now ***

Go to: **Runtime > Restart session** (or press Ctrl+M .), then continue from BLOCK B below.

Do **not** run any cells below this point until after the runtime has restarted.

After restart, continue in **this notebook** — run the BLOCK B cells below.

---